# GLS Lesion Segmentation – TPU Multi-Run Training

**Runtime → Change runtime type → TPU** before running this notebook.

This notebook:
1. Installs `torch_xla` (Google's PyTorch-on-TPU library)
2. Uploads the project zip directly into the Colab VM (`/content/`)
3. Installs all Python dependencies
4. Runs **4 experiments × 10 seeds = 40 training runs** on the TPU
5. Saves checkpoints, per-epoch CSVs, and an aggregate statistics table
6. Downloads the results back to your machine when done

**Experiments**
| ID | Model | Augmentation |
|---|---|---|
| exp01 | U-Net | No |
| exp02 | U-Net | Yes |
| exp03 | Attention U-Net | No |
| exp04 | Attention U-Net | Yes |

## 0 · Verify TPU is attached

In [ ]:
import os
assert os.environ.get('COLAB_TPU_ADDR'), (
    '❌  No TPU detected.  Go to Runtime → Change runtime type → TPU and re-run.'
)
print('✅  TPU address:', os.environ['COLAB_TPU_ADDR'])

## 1 · Install torch_xla

> **This cell takes ~3 min on first run.**

In [ ]:
import subprocess, sys
import torch

print(f'PyTorch version: {torch.__version__}')

subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'torch_xla[tpu]',
    '-f', 'https://storage.googleapis.com/libtpu-releases/index.html',
    '-q'
], check=True)

import torch_xla.core.xla_model as xm
device = xm.xla_device()
print(f'✅  torch_xla loaded.  XLA device: {device}')

## 2 · Upload the project into the Colab VM

**Option A – Upload a zip from your computer (recommended)**  
Run the cell below. A file picker will appear. Select `gls-lesion-segmentation.zip`
(zip the entire project folder first). The cell unpacks it automatically.

**Option B – Clone from GitHub**  
Skip this cell and run the GitHub cell instead.

In [ ]:
# ── Option A: Upload a zip from your local machine ────────────────────────────
import os, pathlib, zipfile
from google.colab import files

print('Select your gls-lesion-segmentation.zip …')
uploaded = files.upload()   # opens a file picker in the browser

zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content/')

# Find the extracted project root (handles both zip-with and zip-without a top folder)
candidates = [p for p in pathlib.Path('/content').iterdir()
              if p.is_dir() and 'gls' in p.name.lower()]
if not candidates:
    # Zip extracted files directly into /content — use it as root
    PROJECT_ROOT = '/content'
else:
    PROJECT_ROOT = str(candidates[0])

os.chdir(PROJECT_ROOT)
print(f'✅  Project root: {os.getcwd()}')
print('    Contents:', sorted(os.listdir('.')))

In [ ]:
# ── Option B: Clone from GitHub (fill in your repo URL) ──────────────────────
# Uncomment and run this cell INSTEAD of Option A if your project is on GitHub.

# GITHUB_REPO = 'https://github.com/YOUR_USERNAME/gls-lesion-segmentation.git'
# import subprocess, os
# subprocess.run(['git', 'clone', GITHUB_REPO, '/content/gls-lesion-segmentation'], check=True)
# os.chdir('/content/gls-lesion-segmentation')
# print('✅  Cloned to:', os.getcwd())

## 3 · Install project dependencies

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    check=True
)
print('✅  Requirements installed.')

## 4 · Smoke-test: load both models on TPU

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torch_xla.core.xla_model as xm
from src.models.unet import UNet
from src.models.attention_unet import AttentionUNet

device = xm.xla_device()

for Model, name in [(UNet, 'UNet'), (AttentionUNet, 'AttentionUNet')]:
    m = Model(in_channels=3, out_channels=1, base_filters=64, depth=4).to(device)
    x = torch.randn(2, 3, 512, 512).to(device)
    y = m(x)
    xm.mark_step()
    n = sum(p.numel() for p in m.parameters())
    print(f'✅  {name:20s}  in={tuple(x.shape)}  out={tuple(y.shape)}  params={n:,}')

## 5 · Training configuration

In [ ]:
N_RUNS      = 10       # ≥10 recommended by supervisor
BASE_SEED   = 0        # seeds: 0, 1, 2, … N_RUNS-1
MAX_EPOCHS  = 100      # set to None to use config default
PATIENCE    = 999999   # disabled early stopping (train full epochs)

EXPERIMENTS = [
    'exp01_unet_noaug',
    'exp02_unet_aug',
    'exp03_attnunet_noaug',
    'exp04_attnunet_aug',
]

SEEDS = list(range(BASE_SEED, BASE_SEED + N_RUNS))
print(f'Jobs: {len(EXPERIMENTS)} experiments × {N_RUNS} seeds = {len(EXPERIMENTS)*N_RUNS} total')
print(f'Seeds: {SEEDS}')

## 6 · Run all 40 training jobs on TPU

Each job calls `scripts/train.py` with `GLS_DEVICE=xla` in the environment,
which tells the `Trainer` to use `xm.xla_device()` and call `xm.mark_step()`
after every optimizer step.

Results are written incrementally to `outputs/results/summary.csv`.

In [ ]:
import csv, statistics, subprocess, sys, os, pathlib
from datetime import datetime

ROOT        = pathlib.Path.cwd()
RESULTS_DIR = ROOT / 'outputs' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def best_dice_from_log(log_path):
    if not log_path.exists():
        return None
    try:
        with log_path.open(newline='', encoding='utf-8') as f:
            rows = list(csv.DictReader(f))
        dices = [float(r['dice']) for r in rows if r.get('dice')]
        return max(dices) if dices else None
    except Exception:
        return None


all_rows   = []
total      = len(EXPERIMENTS) * N_RUNS
job        = 0
start_time = datetime.now()

for experiment in EXPERIMENTS:
    for seed_index, seed in enumerate(SEEDS):
        job += 1
        tag     = f'seed{seed_index:02d}'
        elapsed = int((datetime.now() - start_time).total_seconds()) // 60
        print(f'\n[Job {job}/{total}  elapsed={elapsed}min]  {experiment}/{tag}  seed={seed}', flush=True)

        cmd = [
            sys.executable,
            str(ROOT / 'scripts' / 'train.py'),
            '--experiment', experiment,
            '--seed',       str(seed),
            '--output-tag', tag,
        ]
        if MAX_EPOCHS is not None:
            cmd += ['--max-epochs', str(MAX_EPOCHS)]
        if PATIENCE is not None:
            cmd += ['--patience', str(PATIENCE)]

        env               = os.environ.copy()
        env['GLS_DEVICE'] = 'xla'   # ← TPU backend

        result  = subprocess.run(cmd, env=env)
        success = result.returncode == 0

        log_path  = ROOT / 'outputs' / 'logs' / f'{experiment}_{tag}.csv'
        best_dice = best_dice_from_log(log_path)

        row = {
            'experiment':    experiment,
            'seed_index':    seed_index,
            'seed':          seed,
            'output_tag':    tag,
            'success':       success,
            'best_val_dice': best_dice if best_dice is not None else 'N/A',
        }
        all_rows.append(row)

        # Incremental save
        with (RESULTS_DIR / 'summary.csv').open('w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(all_rows[0].keys()))
            writer.writeheader()
            writer.writerows(all_rows)

        status   = '✅' if success else '❌'
        dice_str = f'{best_dice:.4f}' if best_dice is not None else 'N/A'
        print(f'  {status}  best_val_dice={dice_str}', flush=True)

print('\n🏁 All jobs finished.')

## 7 · Aggregate statistics (mean ± std per experiment)

In [ ]:
from collections import defaultdict
import pandas as pd

by_exp = defaultdict(list)
for row in all_rows:
    val = row['best_val_dice']
    if isinstance(val, float):
        by_exp[row['experiment']].append(val)

agg_rows = []
for exp in EXPERIMENTS:
    dices = by_exp.get(exp, [])
    if not dices:
        agg_rows.append({'experiment': exp, 'n': 0,
                         'mean_dice': None, 'std_dice': None,
                         'min_dice': None,  'max_dice': None})
        continue
    agg_rows.append({
        'experiment': exp,
        'n':          len(dices),
        'mean_dice':  round(statistics.mean(dices), 6),
        'std_dice':   round(statistics.stdev(dices) if len(dices) > 1 else 0.0, 6),
        'min_dice':   round(min(dices), 6),
        'max_dice':   round(max(dices), 6),
    })

agg_df   = pd.DataFrame(agg_rows)
agg_path = RESULTS_DIR / 'aggregate.csv'
agg_df.to_csv(agg_path, index=False)

print('📊 Aggregate results (best-val-Dice across seeds)')
display(agg_df.set_index('experiment'))
print(f'\nSaved → {agg_path}')

## 8 · Per-run summary

In [ ]:
import pandas as pd
summary_df = pd.read_csv(RESULTS_DIR / 'summary.csv')
display(summary_df)

## 9 · Download all outputs back to your machine

The Colab VM is ephemeral — everything is lost when the session ends.
This cell zips the entire `outputs/` folder and triggers a browser download.

In [ ]:
import pathlib, shutil, os
from google.colab import files

ROOT       = pathlib.Path.cwd()
OUTPUT_DIR = ROOT / 'outputs'
ZIP_PATH   = '/content/gls_outputs.zip'

print('Zipping outputs/ …')
shutil.make_archive('/content/gls_outputs', 'zip', ROOT, 'outputs')
size_mb = os.path.getsize(ZIP_PATH) / 1_048_576
print(f'Archive size: {size_mb:.1f} MB')

print('Downloading …')
files.download(ZIP_PATH)   # triggers a browser download dialog
print('✅  Download started.')

## 10 · (Optional) List all output files

In [ ]:
ROOT = pathlib.Path.cwd()
print('Output files in VM:')
for p in sorted((ROOT / 'outputs').rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size // 1024
        print(f'  {p.relative_to(ROOT)}  ({size_kb} KB)')